### Compute jaccard similarity between DEG sets

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
import os
import pickle

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

from collections import Counter
from upsetplot import UpSet
from scipy import stats
import gseapy as gp
from gseapy import barplot, dotplot

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import time as time

from itertools import combinations
from itertools import product

In [54]:
plots_dir = "disease_intersection_plots/"
os.makedirs(plots_dir, exist_ok=True)

In [3]:
def get_significant_genes(df, padj_threshold=0.05, lfc_threshold=0.5):
    sig = df[(df['padj'] < padj_threshold) & (df['log2FoldChange'].abs() > lfc_threshold)]
    
    up_genes = sig[sig['log2FoldChange'] > 0].index
    down_genes = sig[sig['log2FoldChange'] < 0].index
    
    return up_genes, down_genes

In [33]:
def jaccard(set1, set2):
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0

In [4]:
def load_DEG_df(contrast, cell_type):
    '''Load in the DEG for the specified contrast'''

    if contrast == "DCM":
        df = ( pd.read_csv("../non_binarized/pydeseq2_results/" + cell_type + "_disease_DCM_vs_ND_results.csv", 
                           index_col = 0) )
    elif contrast == "ICM":
        df = ( pd.read_csv("../non_binarized/pydeseq2_results/" + cell_type + "_disease_ICM_vs_ND_results.csv", 
                               index_col = 0) )
    elif contrast == "HCM": 
        df = ( pd.read_csv("../non_binarized/pydeseq2_results/" + cell_type + "_disease_HCM_vs_ND_results.csv", 
                               index_col = 0) )
    return df

In [53]:
cell_types = ["Cardiomyocyte", "Endothelial", "Fibroblast", "LEC", "Lymphoid", "Myeloid", "Neuronal", "Pericyte"]

In [58]:
jaccard_results = []

for cell_type in cell_types:
    
    # load DEG dfs
    DCM_df = pd.read_csv(f"../non_binarized/pydeseq2_results/{cell_type}_disease_DCM_vs_ND_results.csv", index_col=0)
    HCM_df = pd.read_csv(f"../non_binarized/pydeseq2_results/{cell_type}_disease_HCM_vs_ND_results.csv", index_col=0)
    ICM_df = pd.read_csv(f"../non_binarized/pydeseq2_results/{cell_type}_disease_ICM_vs_ND_results.csv", index_col=0)

    # make sets of up/down DEGs
    deg_sets_up = {
        "ICM": set(ICM_df.loc[(ICM_df['significant'] == True) & (ICM_df['log2FoldChange'] > 0), 'gene_id']),
        "DCM": set(DCM_df.loc[(DCM_df['significant'] == True) & (DCM_df['log2FoldChange'] > 0), 'gene_id']),
        "HCM": set(HCM_df.loc[(HCM_df['significant'] == True) & (HCM_df['log2FoldChange'] > 0), 'gene_id'])
    }

    deg_sets_down = {
        "ICM": set(ICM_df.loc[(ICM_df['significant'] == True) & (ICM_df['log2FoldChange'] < 0), 'gene_id']),
        "DCM": set(DCM_df.loc[(DCM_df['significant'] == True) & (DCM_df['log2FoldChange'] < 0), 'gene_id']),
        "HCM": set(HCM_df.loc[(HCM_df['significant'] == True) & (HCM_df['log2FoldChange'] < 0), 'gene_id'])
    }

    diseases = list(deg_sets_up.keys())

    # compute pairwise Jaccard
    for d1, d2 in combinations(diseases, 2):
        jacc_up = jaccard(deg_sets_up[d1], deg_sets_up[d2])
        jacc_down = jaccard(deg_sets_down[d1], deg_sets_down[d2])

        jaccard_results.append({
            "cell_type": cell_type,
            "disease1": d1,
            "disease2": d2,
            "direction": "up",
            "jaccard": jacc_up
        })
        jaccard_results.append({
            "cell_type": cell_type,
            "disease1": d1,
            "disease2": d2,
            "direction": "down",
            "jaccard": jacc_down
        })

# convert list to df
jaccard_df = pd.DataFrame(jaccard_results)

In [59]:
jaccard_df

,cell_type,disease1,disease2,direction,jaccard
0,Cardiomyocyte,ICM,DCM,up,0.136600
1,Cardiomyocyte,ICM,DCM,down,0.121612
2,Cardiomyocyte,ICM,HCM,up,0.150920
3,Cardiomyocyte,ICM,HCM,down,0.165323
4,Cardiomyocyte,DCM,HCM,up,0.266346
5,Cardiomyocyte,DCM,HCM,down,0.268307
6,Endothelial,ICM,DCM,up,0.131425
7,Endothelial,ICM,DCM,down,0.112309
8,Endothelial,ICM,HCM,up,0.178005
9,Endothelial,ICM,HCM,down,0.234932
